In [2]:
!pip install sentence-transformers --quiet
!pip install faiss-cpu --quiet


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load the embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dim = embedding_model.get_sentence_embedding_dimension()

/home/javierdlrm/.pyenv/versions/3.10.14/envs/ik2221/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class RAG:
    def __init__(self, embedding_model, embedding_dim):
        self.embedding_model = embedding_model
        self.embedding_dim = embedding_dim
        self.faiss_index = faiss.IndexFlatIP(embedding_dim)  # Use inner product for cosine
        self.context_keys = []
        self.contexts = {}

    def index(self, context_key, context):
        print("/// [RAG] indexing context: " + context_key)
        embeddings_np = self.embedding_model.encode([context], normalize_embeddings=True)
        self.faiss_index.add(embeddings_np.astype(np.float32))
        self.context_keys.append(context_key)
        self.contexts[context_key] = context

    def search(self, question, top_k=5):
        print("/// [RAG] searching for question: " + question)
        question_np = self.embedding_model.encode([question], normalize_embeddings=True)
        distances, ids = self.faiss_index.search(question_np.astype(np.float32), top_k)
        context_key = self.context_keys[ids[0][0]]
        context = self.contexts[context_key]
        return context_key, context

In [8]:
rag_instance = RAG(embedding_model=embedding_model, embedding_dim=embedding_dim)

In [6]:
import os

data_dir = "../frontend/data/"
for filename in os.listdir(data_dir):
    if filename.endswith(".txt"):
        file_path = os.path.join(data_dir, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            rag_instance.index(context_key=filename, context=content)

/// [RAG] indexing context: sigcomm2023_janus.txt
/// [RAG] indexing context: cacheblend.txt
/// [RAG] indexing context: extra-compound-mape.txt
/// [RAG] indexing context: extra-biology.txt
/// [RAG] indexing context: extra-estimate-performance.txt
/// [RAG] indexing context: osdi24-lee.txt
/// [RAG] indexing context: extra-leonardo.txt
/// [RAG] indexing context: SplitRPC-sigmetrics23.txt
/// [RAG] indexing context: extra-finance.txt
/// [RAG] indexing context: aplomb-sigcomm12.txt
/// [RAG] indexing context: vllm.txt
/// [RAG] indexing context: click.txt
/// [RAG] indexing context: extra-music.txt
/// [RAG] indexing context: nsdi20-paper-barbette.txt
/// [RAG] indexing context: ServerlessLLM_summary.txt
/// [RAG] indexing context: sigcomm24-crux.txt
/// [RAG] indexing context: osdi24-sun-biao.txt
/// [RAG] indexing context: nsdi22-paper-reda_1.txt
/// [RAG] indexing context: osdi24-agrawal.txt
/// [RAG] indexing context: extra-oss-monitoring.txt
/// [RAG] indexing context: metron-ns

In [ ]:
import os

prompts_dir = "../frontend/prompts/"
accuracy = 0
total = 0
prompt_files = os.listdir(prompts_dir)
for prompt_filename in prompt_files:
    if prompt_filename.endswith(".txt"):
        prompt_path = os.path.join(prompts_dir, prompt_filename)
        with open(prompt_path, "r", encoding="utf-8") as prompt_file:
            for line in prompt_file:
                question = line.strip()
                if not question:
                    continue
                context_key, _ = rag_instance.search(question=question)
                if prompt_filename == context_key:
                    accuracy += 1
                print(f"////// Matched: {prompt_filename == context_key}")
                total += 1

if total > 0:
    print(f"# Accuracy: {(accuracy / total) * 100}%")
else:
    print("# No questions found.")

/// [RAG] searching for question: What is the main motivation for using decentralized control in self-adaptive systems according to the paper?
//// Distances:  [[0.6217058  0.21929428 0.19829683 0.18997075 0.1785673 ]]
//// IDs:  [[ 1 11 14  7  5]]
////// Matched: True
/// [RAG] searching for question: What is the definition of epigenetics?
//// Distances:  [[0.6104646  0.0752614  0.07387325 0.04357736 0.03292998]]
//// IDs:  [[2 0 1 8 9]]
////// Matched: True
/// [RAG] searching for question: What is the main problem addressed by the PAPE algorithm in the context of machine learning model deployment?
//// Distances:  [[0.52827495 0.3198614  0.22368547 0.2180513  0.20323795]]
//// IDs:  [[ 3  8 12  5 15]]
////// Matched: True
/// [RAG] searching for question: Who is described as the quintessential Renaissance man in the report?
//// Distances:  [[0.49107942 0.13006036 0.05337492 0.04378686 0.04364811]]
//// IDs:  [[ 4  0  6 12 10]]
////// Matched: True
/// [RAG] searching for question: